In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import threading
import queue

# Do przekazywania obrazu używałem na telefonie aplikacji IP webcam na androidzie
STREAM_URL = "http://192.168.135.9:8080/video"

MAX_WIDTH = 900
JPEG_QUALITY = 50
FPS = 15 
TIMEOUT = 5 # zatrzymuje kamerke po X sekundach

# TODO:
# implementacja tego ale zamiast łączenia z kamerką to danie po prostu scierzki do video
# analiza obrazu wg wymagań projektu
# gl hf 


stop = threading.Event()
q = queue.Queue(maxsize=1)
img_widget = widgets.Image(format='jpeg')


def resize(frame):
	h, w = frame.shape[:2]
	if w > MAX_WIDTH:
		return cv2.resize(frame, (MAX_WIDTH, int(h * MAX_WIDTH / w)))
	return frame


def analyze(frame):
	frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
	frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
	return frame


def fetch():
	cap = cv2.VideoCapture(STREAM_URL)
	cap.set(cv2.CAP_PROP_FPS, FPS)
	while not stop.is_set():
		ret, frame = cap.read()
		if ret:
			try:
				q.put_nowait(resize(frame))
			except queue.Full:
				pass
	cap.release()


def process():
	while not stop.is_set():
		try:
			frame = q.get(timeout=1)
			frame = analyze(frame)
			_, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
			img_widget.value = buf.tobytes()
		except:
			pass


def start():
	display(img_widget)
	threading.Thread(target=fetch, daemon=True).start()
	threading.Thread(target=process, daemon=True).start()
	threading.Timer(TIMEOUT, stop.set).start()

start()

Image(value=b'', format='jpeg')

In [9]:
stop.set()